# 🎯 Siratify — Système de Recommandation v2
**Approche :** Content-Based Filtering — Comparaison TF-IDF / BM25 / CountVectorizer  
**Métriques :** Precision@K · Recall@K · Coverage · Personalization  
**Extras :** Gestion Cold Start  

| Dataset | Taille | Source |
|---------|--------|--------|
| `users.csv` | 500 utilisateurs | Inspiré de [LinkedIn Job Postings — Kaggle](https://www.kaggle.com/datasets/arshkon/linkedin-job-postings) |
| `content.csv` | 300 posts | Inspiré de [Medium Articles — Kaggle](https://www.kaggle.com/datasets/fabiochiusano/medium-articles) |

> Mettre les 3 fichiers dans le même dossier puis faire **Run All**.

## 📦 1. Imports & Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, math, random
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from collections import defaultdict

warnings.filterwarnings('ignore')
random.seed(42)
np.random.seed(42)

# ── BM25 implémenté nativement (sans dépendance externe) ────────────────────
class BM25:
    """
    BM25 (Best Match 25) — Robertson et al. 1994.
    Amélioration du TF-IDF : saturation TF (k1) + normalisation longueur (b).
    """
    def __init__(self, k1=1.5, b=0.75):
        self.k1, self.b = k1, b

    def fit(self, corpus):
        self.docs_   = [doc.split() for doc in corpus]
        self.N_      = len(self.docs_)
        self.avgdl_  = np.mean([len(d) for d in self.docs_])
        df = defaultdict(int)
        for doc in self.docs_:
            for w in set(doc): df[w] += 1
        self.idf_ = {w: math.log((self.N_-n+0.5)/(n+0.5)+1) for w,n in df.items()}
        return self

    def transform(self, queries):
        scores = np.zeros((len(queries), self.N_))
        for qi, query in enumerate(queries):
            q_terms = query.split()
            for di, doc in enumerate(self.docs_):
                dl, tf_map, score = len(doc), defaultdict(int), 0.0
                for w in doc: tf_map[w] += 1
                for t in q_terms:
                    if t not in self.idf_: continue
                    tf = tf_map.get(t, 0)
                    score += self.idf_[t]*tf*(self.k1+1) / (tf+self.k1*(1-self.b+self.b*dl/self.avgdl_))
                scores[qi, di] = score
        return scores

print('✅ Imports OK  |  pandas', pd.__version__, '| numpy', np.__version__)

## 📂 2. Chargement des données (500 users · 300 posts)

In [ ]:
users_df   = pd.read_csv('users.csv').reset_index(drop=True)
content_df = pd.read_csv('content.csv').reset_index(drop=True)
print(f'✅ Utilisateurs : {len(users_df)}')
print(f'✅ Contenus     : {len(content_df)}')

In [ ]:
print('=== Aperçu users.csv ===')
display(users_df.head(5))

In [ ]:
print('=== Aperçu content.csv ===')
display(content_df.head(5))

## 📊 3. Analyse Exploratoire des Données (EDA)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

sector_counts = users_df['business_activity'].value_counts()
axes[0,0].barh(sector_counts.index, sector_counts.values, color='#1B4F8A', alpha=0.85)
axes[0,0].set_title('Distribution des secteurs (users)', fontsize=12, fontweight='bold')
for i, v in enumerate(sector_counts.values): axes[0,0].text(v+0.5, i, str(v), va='center', fontsize=9)

role_counts = users_df['role'].value_counts().head(10)
axes[0,1].barh(role_counts.index, role_counts.values, color='#2980B9', alpha=0.85)
axes[0,1].set_title('Top 10 rôles professionnels', fontsize=12, fontweight='bold')

type_counts = content_df['type'].value_counts()
colors_pie = ['#1B4F8A','#2980B9','#5DADE2','#85C1E9','#AED6F1']
axes[1,0].pie(type_counts.values, labels=type_counts.index, autopct='%1.1f%%',
              colors=colors_pie[:len(type_counts)], startangle=140)
axes[1,0].set_title('Types de contenus', fontsize=12, fontweight='bold')

users_df['n_interests'] = users_df['interests'].apply(lambda x: len(str(x).split(',')))
axes[1,1].hist(users_df['n_interests'], bins=range(2,9), color='#27AE60', alpha=0.8, edgecolor='white')
axes[1,1].set_title("Nombre d'intérêts par utilisateur", fontsize=12, fontweight='bold')
axes[1,1].set_xlabel("Nombre d'intérêts"); axes[1,1].set_ylabel("Nombre d'utilisateurs")

plt.suptitle("Analyse Exploratoire — Siratify (500 users · 300 posts)", fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig1_eda.png', dpi=150, bbox_inches='tight')
plt.show()
print('📸 fig1_eda.png sauvegardé')

## 🔧 4. Prétraitement des données

In [ ]:
def preprocess_content(df):
    df = df.copy()
    for c in ['title','tags','type']: df[c] = df[c].fillna('')
    df['tags_clean'] = df['tags'].str.replace(',', ' ')
    df['combined_features'] = (df['title']+' '+df['title']+' '+df['tags_clean']+' '+df['type']).str.lower()
    return df

def preprocess_users(df):
    df = df.copy()
    for c in ['interests','role','business_activity']: df[c] = df[c].fillna('')
    ic = df['interests'].str.replace(',', ' ')
    df['user_profile'] = (ic+' '+ic+' '+df['role']+' '+df['business_activity']).str.lower()
    return df

content_df = preprocess_content(content_df)
users_df   = preprocess_users(users_df)

print('✅ Prétraitement terminé')
print('\nExemple profil user (U0001):')
print(users_df.iloc[0]['user_profile'])
print('\nExemple combined_features (C0001):')
print(content_df.iloc[0]['combined_features'])

## 🔢 5. Vectorisation — TF-IDF · BM25 · CountVectorizer

| Méthode | Caractéristique principale |
|---------|---------------------------|
| **TF-IDF** | Pondération par rareté du terme dans le corpus (IDF) |
| **BM25** | Saturation de la fréquence + normalisation par longueur du document |
| **CountVectorizer** | Fréquence brute, sans pondération IDF |

In [ ]:
corpus_content = list(content_df['combined_features'])
corpus_users   = list(users_df['user_profile'])
corpus_all     = corpus_content + corpus_users

# TF-IDF
tfidf_vec = TfidfVectorizer(max_features=8000, ngram_range=(1,2), stop_words='english', sublinear_tf=True)
tfidf_vec.fit(corpus_all)
content_tfidf = tfidf_vec.transform(corpus_content)
user_tfidf    = tfidf_vec.transform(corpus_users)
sim_tfidf     = cosine_similarity(user_tfidf, content_tfidf)

# BM25
bm25 = BM25(k1=1.5, b=0.75)
bm25.fit(corpus_content)
sim_bm25_raw = bm25.transform(corpus_users)
bm25_max = sim_bm25_raw.max(axis=1, keepdims=True)
bm25_max[bm25_max == 0] = 1
sim_bm25 = sim_bm25_raw / bm25_max

# CountVectorizer
count_vec = CountVectorizer(max_features=8000, ngram_range=(1,2), stop_words='english')
count_vec.fit(corpus_all)
content_count = count_vec.transform(corpus_content)
user_count    = count_vec.transform(corpus_users)
sim_count     = cosine_similarity(user_count, content_count)

print('✅ TF-IDF   matrice :', sim_tfidf.shape)
print('✅ BM25     matrice :', sim_bm25.shape)
print('✅ CountVec matrice :', sim_count.shape)

## 🧪 6. Simulation d'interactions (ground truth pour Precision/Recall)

Un post est **pertinent** pour un user si au moins un de ses intérêts apparaît dans les tags du post.
On simule 5–20 interactions par utilisateur avec 10% de bruit aléatoire.

In [ ]:
def simulate_interactions(users_df, content_df, n_min=5, n_max=20, noise=0.1, seed=42):
    rng = np.random.RandomState(seed)
    gt  = {}
    for _, user in users_df.iterrows():
        uid       = user['user_id']
        interests = set(i.strip().lower() for i in str(user['interests']).split(','))
        sector    = user['business_activity'].lower()
        rel = [ci for ci, post in content_df.iterrows()
               if interests & set(t.strip().lower() for t in str(post['tags']).split(','))
               or sector in ' '.join(str(post['tags']).lower().split(','))]
        n_seen = rng.randint(n_min, max(n_min+1, min(n_max+1, len(rel)+2)))
        seen   = list(rng.choice(rel, size=min(n_seen, len(rel)), replace=False)) if rel else []
        non_rel = [i for i in range(len(content_df)) if i not in seen]
        n_noise = int(len(seen)*noise)
        if n_noise > 0 and len(non_rel) >= n_noise:
            seen += list(rng.choice(non_rel, size=n_noise, replace=False))
        gt[uid] = set(seen)
    return gt

ground_truth = simulate_interactions(users_df, content_df)
sizes = [len(v) for v in ground_truth.values()]
print(f'✅ Ground truth : {len(ground_truth)} users | moy {np.mean(sizes):.1f} posts/user')

## 📐 7. Fonctions de métriques

$$\text{Precision@K} = \frac{|\text{Rec@K} \cap \text{Pertinents}|}{K} \qquad
\text{Recall@K} = \frac{|\text{Rec@K} \cap \text{Pertinents}|}{|\text{Pertinents}|}$$

In [ ]:
def precision_at_k(rec_idx, relevant, k):
    return len(set(rec_idx[:k]) & relevant) / k

def recall_at_k(rec_idx, relevant, k):
    if not relevant: return 0.0
    return len(set(rec_idx[:k]) & relevant) / len(relevant)

def evaluate_model(sim_matrix, users_df, content_df, ground_truth, name, K=10, sample_n=200):
    np.random.seed(42)
    uids = list(np.random.choice(users_df['user_id'].tolist(), size=min(sample_n, len(users_df)), replace=False))
    p_scores, r_scores, all_recs, per_user = [], [], set(), {}
    for uid in uids:
        uidx    = users_df[users_df['user_id']==uid].index[0]
        rec_idx = list(np.argsort(sim_matrix[uidx])[::-1][:K])
        rel     = ground_truth.get(uid, set())
        p_scores.append(precision_at_k(rec_idx, rel, K))
        r_scores.append(recall_at_k(rec_idx, rel, K))
        all_recs.update(rec_idx)
        per_user[uid] = set(rec_idx)
    coverage = len(all_recs)/len(content_df)*100
    uid_list = list(per_user.keys())
    overlaps = [len(per_user[uid_list[i]]&per_user[uid_list[j]])/len(per_user[uid_list[i]]|per_user[uid_list[j]])
                for i in range(min(len(uid_list),40)) for j in range(i+1,min(len(uid_list),40))
                if per_user[uid_list[i]]|per_user[uid_list[j]]]
    personalization = (1-np.mean(overlaps))*100 if overlaps else 0
    return {'Modèle': name, f'Precision@{K}': round(np.mean(p_scores),4),
            f'Recall@{K}': round(np.mean(r_scores),4),
            'Coverage (%)': round(coverage,1), 'Personalization (%)': round(personalization,1)}

print('✅ Fonctions de métriques prêtes')

## ⚖️ 8. Comparaison TF-IDF vs BM25 vs CountVectorizer

In [ ]:
print('Évaluation en cours...')

def eval_(sim, name, K):
    return evaluate_model(sim, users_df, content_df, ground_truth, name, K=K)

r10 = [eval_(sim_tfidf,'TF-IDF',10), eval_(sim_bm25,'BM25',10), eval_(sim_count,'CountVectorizer',10)]
r5  = [eval_(sim_tfidf,'TF-IDF',5),  eval_(sim_bm25,'BM25',5),  eval_(sim_count,'CountVectorizer',5)]

df_k10 = pd.DataFrame(r10).set_index('Modèle')
df_k5  = pd.DataFrame(r5).set_index('Modèle')

print('=== Résultats K=10 ===')
display(df_k10.style.highlight_max(color='#d4edda').format(precision=4))
print('\n=== Résultats K=5 ===')
display(df_k5.style.highlight_max(color='#d4edda').format(precision=4))

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
models_list = df_k10.index.tolist()
colors_bar  = ['#1B4F8A','#E67E22','#27AE60']
metrics_list = ['Precision@10','Recall@10','Coverage (%)','Personalization (%)']

for ax, metric in zip(axes, metrics_list):
    vals = df_k10[metric].tolist()
    bars = ax.bar(models_list, vals, color=colors_bar, alpha=0.85)
    ax.set_title(metric, fontsize=11, fontweight='bold')
    ax.set_ylim(0, max(vals)*1.3 if max(vals)>0 else 1)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+max(vals)*0.02,
                f'{val:.3f}' if val<10 else f'{val:.1f}%', ha='center', fontsize=9, fontweight='bold')
    ax.set_xticklabels(models_list, rotation=15, ha='right')
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

plt.suptitle('Comparaison TF-IDF vs BM25 vs CountVectorizer — K=10', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig2_comparaison_modeles.png', dpi=150, bbox_inches='tight')
plt.show()
print('📸 fig2_comparaison_modeles.png sauvegardé')

## 📈 9. Courbes Precision@K et Recall@K

In [ ]:
K_values = [1, 3, 5, 10, 15, 20]
curve_models = [('TF-IDF', sim_tfidf,'#1B4F8A','-o'),
                ('BM25',   sim_bm25, '#E67E22','-s'),
                ('Count',  sim_count,'#27AE60','-^')]

sample_uids = users_df['user_id'].sample(150, random_state=42).tolist()
prec_curves = {m[0]:[] for m in curve_models}
rec_curves  = {m[0]:[] for m in curve_models}

for K in K_values:
    for mname, sim_mat, _, _ in curve_models:
        ps, rs = [], []
        for uid in sample_uids:
            uidx    = users_df[users_df['user_id']==uid].index[0]
            rec_idx = list(np.argsort(sim_mat[uidx])[::-1][:K])
            rel     = ground_truth.get(uid, set())
            ps.append(precision_at_k(rec_idx, rel, K))
            rs.append(recall_at_k(rec_idx, rel, K))
        prec_curves[mname].append(np.mean(ps))
        rec_curves[mname].append(np.mean(rs))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
for mname, _, color, ls in curve_models:
    ax1.plot(K_values, prec_curves[mname], ls, color=color, label=mname, linewidth=2.5, markersize=8)
    ax2.plot(K_values, rec_curves[mname],  ls, color=color, label=mname, linewidth=2.5, markersize=8)

for ax, title in [(ax1,'Precision@K'),(ax2,'Recall@K')]:
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('K'); ax.legend(); ax.grid(alpha=0.3)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

plt.suptitle('Courbes Precision@K et Recall@K', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig3_precision_recall_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('📸 fig3_precision_recall_curves.png sauvegardé')

## 🏆 10. Sélection du meilleur modèle & SiratifyRecommender

In [ ]:
df_score = df_k10.copy()
df_score['Score_composite'] = (0.40*df_score['Precision@10'] + 0.30*df_score['Recall@10'] +
    0.15*df_score['Coverage (%)']/100 + 0.15*df_score['Personalization (%)']/100).round(4)

best_model_name = df_score['Score_composite'].idxmax()
SIM_MAP = {'TF-IDF': sim_tfidf, 'BM25': sim_bm25, 'CountVectorizer': sim_count}
best_sim = SIM_MAP[best_model_name]
print('=== Score composite ===' )
display(df_score)
print(f'\n🏆 Meilleur modèle : {best_model_name}')

## 🤖 11. Classe SiratifyRecommender (avec Cold Start)

In [ ]:
class SiratifyRecommender:
    """
    Content-Based Recommender pour Siratify.
    Fonctionnalités :
      - recommend()             : recommandations pour utilisateur existant
      - recommend_cold_start()  : recommandations pour nouvel utilisateur (sans historique)
      - get_similar_posts()     : posts similaires à un post donné
    """
    def __init__(self, users_df, content_df, sim_matrix, tfidf_vec, model_name='Model'):
        self.users_df    = users_df.reset_index(drop=True)
        self.content_df  = content_df.reset_index(drop=True)
        self.sim_matrix  = sim_matrix
        self.tfidf_vec   = tfidf_vec
        self.model_name  = model_name
        self.content_vecs = tfidf_vec.transform(content_df['combined_features'])
        print(f'✅ Recommandeur [{model_name}] prêt — matrice {sim_matrix.shape}')

    def recommend(self, user_id, top_n=10, diversity_factor=0.2):
        mask = self.users_df['user_id'] == user_id
        if not mask.any(): raise ValueError(f"'{user_id}' introuvable — utiliser recommend_cold_start()")
        uidx  = self.users_df[mask].index[0]
        return self.users_df.loc[uidx], self._build(self.sim_matrix[uidx], top_n, diversity_factor)

    def recommend_cold_start(self, interests, role='', sector='', top_n=10):
        """Nouvel utilisateur : profil textuel → vectorisation directe → recommandation."""
        profile = (interests+' '+interests+' '+role+' '+sector).lower()
        vec     = self.tfidf_vec.transform([profile])
        scores  = cosine_similarity(vec, self.content_vecs).flatten()
        recs    = self._build(scores, top_n, diversity_factor=0.1)
        info    = pd.Series({'user_id':'NEW','full_name':'Nouvel utilisateur',
                              'role':role or 'N/A','interests':interests,
                              'business_activity':sector or 'N/A'})
        return info, recs

    def _build(self, scores, top_n, diversity_factor):
        sorted_idx = np.argsort(scores)[::-1]
        n_p  = int(top_n*(1-diversity_factor))
        n_d  = top_n-n_p
        top  = sorted_idx[:n_p]
        rest = sorted_idx[n_p:]
        disc = np.random.choice(rest, size=min(n_d,len(rest)), replace=False) if n_d>0 and len(rest)>0 else []
        disc_set = set(disc)
        rows = []
        for rank, idx in enumerate(list(top)+list(disc), 1):
            p = self.content_df.iloc[idx]
            rows.append({'Rang':rank,'ID':p['content_id'],'Titre':p['title'],
                         'Type':p['type'],'Score':round(float(scores[idx]),4),
                         'Type Rec':'🔍 Découverte' if idx in disc_set else '✅ Personnalisé'})
        return pd.DataFrame(rows)

    def get_similar_posts(self, content_id, top_n=5):
        mask = self.content_df['content_id']==content_id
        if not mask.any(): raise ValueError(f"'{content_id}' introuvable")
        idx = self.content_df[mask].index[0]
        sim = cosine_similarity(self.content_vecs[idx], self.content_vecs).flatten()
        sim[idx] = -1
        top = np.argsort(sim)[::-1][:top_n]
        return pd.DataFrame([{'content_id':self.content_df.iloc[i]['content_id'],
                               'Titre':self.content_df.iloc[i]['title'],
                               'Score':round(float(sim[i]),4)} for i in top])

recommender = SiratifyRecommender(users_df, content_df, best_sim, tfidf_vec, model_name=best_model_name)

## 🎯 12. Recommandations — Utilisateurs existants

In [ ]:
for uid in ['U0001','U0005','U0010','U0020']:
    uinfo, recs = recommender.recommend(uid, top_n=5, diversity_factor=0.2)
    print(f'\n{"="*65}')
    print(f'👤 {uinfo["full_name"]} ({uid})  |  {uinfo["role"]}')
    print(f'⭐ {uinfo["interests"]}')
    print(f'{"="*65}')
    display(recs[['Rang','Titre','Score','Type Rec']])

## ❄️ 13. Cold Start — Nouvel utilisateur sans historique

**Problème :** un nouvel utilisateur n'a aucune interaction — pas de matrice de similarité disponible.  
**Solution ici :** on vectorise directement son profil déclaré avec le même modèle TF-IDF.

In [ ]:
# Scénario 1 — Nouveau Data Scientist
uinfo, recs = recommender.recommend_cold_start(
    interests='machine learning python deep learning NLP statistics',
    role='Data Scientist', sector='Data & Analytics', top_n=5)
print(f'❄️ {uinfo["role"]} | {uinfo["business_activity"]}')
display(recs[['Rang','Titre','Score','Type Rec']])

In [ ]:
# Scénario 2 — Nouveau Marketing Manager
uinfo, recs = recommender.recommend_cold_start(
    interests='SEO content marketing social media digital marketing growth',
    role='Marketing Manager', sector='Marketing & Communication', top_n=5)
print(f'❄️ {uinfo["role"]} | {uinfo["business_activity"]}')
display(recs[['Rang','Titre','Score','Type Rec']])

In [ ]:
# Scénario 3 — Profil minimal (2 intérêts seulement)
uinfo, recs = recommender.recommend_cold_start(interests='startup fundraising', top_n=5)
print(f'❄️ Profil minimal : "{uinfo["interests"]}"')
display(recs[['Rang','Titre','Score','Type Rec']])

In [ ]:
# Visualisation Cold Start vs utilisateur existant
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
uinfo1, recs1 = recommender.recommend('U0001', top_n=5, diversity_factor=0)
t1 = [t[:32]+'...' if len(t)>32 else t for t in recs1['Titre'].tolist()]
ax1.barh(t1[::-1], recs1['Score'].tolist()[::-1], color='#1B4F8A', alpha=0.85)
ax1.set_title(f'Utilisateur existant\n{uinfo1["full_name"]} ({uinfo1["role"]})', fontweight='bold')
ax1.set_xlabel('Score')

_, recs_cs = recommender.recommend_cold_start(
    interests=uinfo1['interests'].replace(',',' '), role=uinfo1['role'],
    sector=uinfo1['business_activity'], top_n=5)
t2 = [t[:32]+'...' if len(t)>32 else t for t in recs_cs['Titre'].tolist()]
ax2.barh(t2[::-1], recs_cs['Score'].tolist()[::-1], color='#E67E22', alpha=0.85)
ax2.set_title(f'Cold Start (même profil)\nRole: {uinfo1["role"]}', fontweight='bold')
ax2.set_xlabel('Score')

plt.suptitle('Recommandations : Utilisateur existant vs Cold Start', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig4_cold_start.png', dpi=150, bbox_inches='tight')
plt.show()
print('📸 fig4_cold_start.png sauvegardé')

## 🗺️ 14. Heatmap de similarité

In [ ]:
fig, ax = plt.subplots(figsize=(18, 8))
subset   = best_sim[:20, :20]
u_labels = [f"{r['full_name'].split()[0]} ({r['role'][:12]})" for _, r in users_df.head(20).iterrows()]
c_labels = [t[:28]+'...' if len(t)>28 else t for t in content_df['title'].tolist()[:20]]
sns.heatmap(subset, xticklabels=c_labels, yticklabels=u_labels, cmap='YlOrRd',
            annot=True, fmt='.2f', linewidths=0.3, cbar_kws={'label':'Score'}, ax=ax)
ax.set_title(f'Matrice de Similarité [{best_model_name}] — 20 Users × 20 Posts',
             fontsize=13, fontweight='bold', pad=15)
plt.xticks(rotation=45, ha='right', fontsize=8); plt.yticks(fontsize=9)
plt.tight_layout()
plt.savefig('fig5_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('📸 fig5_heatmap.png sauvegardé')

## 📊 15. Résumé final

In [ ]:
df_final = pd.merge(
    df_k5.rename(columns={'Precision@5':'P@5','Recall@5':'R@5'})[['P@5','R@5']],
    df_k10.rename(columns={'Precision@10':'P@10','Recall@10':'R@10'})[['P@10','R@10','Coverage (%)','Personalization (%)']],
    left_index=True, right_index=True)
df_final['Score_Composite'] = (0.40*df_final['P@10']+0.30*df_final['R@10']+
    0.15*df_final['Coverage (%)']/100+0.15*df_final['Personalization (%)']/100).round(4)

print('=== TABLEAU COMPARATIF FINAL ===')
display(df_final.style.highlight_max(color='#d4edda').highlight_min(color='#f8d7da').format(precision=4))

best = df_final['Score_Composite'].idxmax()
print(f'\n🏆 Meilleur modèle : {best}')
print(f'   Precision@10  : {df_final.loc[best,"P@10"]:.4f}')
print(f'   Recall@10     : {df_final.loc[best,"R@10"]:.4f}')
print(f'   Coverage      : {df_final.loc[best,"Coverage (%)"]}%')
print(f'   Personalization: {df_final.loc[best,"Personalization (%)"]}%')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 6))
models_f = df_final.index.tolist()
colors_f = ['#1B4F8A','#E67E22','#27AE60']
x = np.arange(len(models_f)); w = 0.35

axes[0].bar(x-w/2, df_final['P@5'], w, label='P@5',  color='#5DADE2', alpha=0.85)
axes[0].bar(x+w/2, df_final['P@10'], w, label='P@10', color='#1B4F8A', alpha=0.85)
axes[0].set_xticks(x); axes[0].set_xticklabels(models_f, rotation=15, ha='right')
axes[0].set_title('Precision@5 vs @10', fontweight='bold'); axes[0].legend()

axes[1].bar(x-w/2, df_final['R@5'], w, label='R@5',  color='#82E0AA', alpha=0.85)
axes[1].bar(x+w/2, df_final['R@10'], w, label='R@10', color='#27AE60', alpha=0.85)
axes[1].set_xticks(x); axes[1].set_xticklabels(models_f, rotation=15, ha='right')
axes[1].set_title('Recall@5 vs @10', fontweight='bold'); axes[1].legend()

bars_f = axes[2].bar(models_f, df_final['Score_Composite'], color=colors_f, alpha=0.85)
axes[2].set_title('Score Composite Global', fontweight='bold')
for bar, v in zip(bars_f, df_final['Score_Composite']):
    axes[2].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.001,
                 f'{v:.4f}', ha='center', fontsize=10, fontweight='bold')
axes[2].set_xticklabels(models_f, rotation=15, ha='right')
for ax in axes: ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

plt.suptitle('Résumé Comparatif — TF-IDF vs BM25 vs CountVectorizer', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig6_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n✅ Toutes les figures générées :')
print('   fig1_eda | fig2_comparaison_modeles | fig3_precision_recall_curves')
print('   fig4_cold_start | fig5_heatmap | fig6_summary')

## 🏅 16. NDCG@K (Normalized Discounted Cumulative Gain)

Le **NDCG** est la métrique standard dans la littérature des systèmes de recommandation.  
Elle récompense les modèles qui placent les items pertinents **en tête** de liste.

$$\text{DCG@K} = \sum_{i=1}^{K} \frac{rel_i}{\log_2(i+1)} \qquad \text{NDCG@K} = \frac{\text{DCG@K}}{\text{IDCG@K}}$$

- $rel_i = 1$ si le post à la position $i$ est pertinent, 0 sinon  
- $\text{IDCG@K}$ = DCG du classement idéal (tous les pertinents en premier)  
- NDCG ∈ [0, 1] — plus proche de 1 = meilleur classement

In [ ]:
def dcg_at_k(rec_idx, relevant, k):
    """Discounted Cumulative Gain à K."""
    score = 0.0
    for i, idx in enumerate(rec_idx[:k], 1):
        if idx in relevant:
            score += 1.0 / math.log2(i + 1)
    return score

def ndcg_at_k(rec_idx, relevant, k):
    """Normalized DCG à K."""
    if not relevant:
        return 0.0
    dcg  = dcg_at_k(rec_idx, relevant, k)
    # Classement idéal : tous les pertinents d'abord
    ideal_hits = min(len(relevant), k)
    idcg = sum(1.0 / math.log2(i + 1) for i in range(1, ideal_hits + 1))
    return dcg / idcg if idcg > 0 else 0.0

def evaluate_ndcg(sim_matrix, users_df, content_df, ground_truth,
                  model_name, K=10, sample_n=200):
    """Évalue Precision@K, Recall@K et NDCG@K en une seule passe."""
    np.random.seed(42)
    uids = list(np.random.choice(users_df['user_id'].tolist(),
                                  size=min(sample_n, len(users_df)), replace=False))
    p_scores, r_scores, n_scores = [], [], []

    for uid in uids:
        uidx    = users_df[users_df['user_id'] == uid].index[0]
        rec_idx = list(np.argsort(sim_matrix[uidx])[::-1][:K])
        rel     = ground_truth.get(uid, set())
        p_scores.append(precision_at_k(rec_idx, rel, K))
        r_scores.append(recall_at_k(rec_idx, rel, K))
        n_scores.append(ndcg_at_k(rec_idx, rel, K))

    return {
        'Modèle'         : model_name,
        f'Precision@{K}' : round(np.mean(p_scores), 4),
        f'Recall@{K}'    : round(np.mean(r_scores), 4),
        f'NDCG@{K}'      : round(np.mean(n_scores), 4),
    }

print('Calcul NDCG en cours...')
ndcg_results = [
    evaluate_ndcg(sim_tfidf, users_df, content_df, ground_truth, 'TF-IDF',         K=10),
    evaluate_ndcg(sim_bm25,  users_df, content_df, ground_truth, 'BM25',            K=10),
    evaluate_ndcg(sim_count, users_df, content_df, ground_truth, 'CountVectorizer', K=10),
]
df_ndcg = pd.DataFrame(ndcg_results).set_index('Modèle')
print('=== Résultats avec NDCG@10 ===')
display(df_ndcg.style.highlight_max(color='#d4edda').format(precision=4))

In [ ]:
# Courbe NDCG@K pour les 3 modèles
K_values_ndcg = [1, 3, 5, 10, 15, 20]
ndcg_curves   = {m: [] for m in ['TF-IDF','BM25','Count']}
sample_uids_n = users_df['user_id'].sample(150, random_state=42).tolist()

for K in K_values_ndcg:
    for mname, sm in [('TF-IDF',sim_tfidf),('BM25',sim_bm25),('Count',sim_count)]:
        scores_n = []
        for uid in sample_uids_n:
            uidx    = users_df[users_df['user_id']==uid].index[0]
            rec_idx = list(np.argsort(sm[uidx])[::-1][:K])
            rel     = ground_truth.get(uid, set())
            scores_n.append(ndcg_at_k(rec_idx, rel, K))
        ndcg_curves[mname].append(np.mean(scores_n))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
colors_ndcg = {'TF-IDF':'#1B4F8A','BM25':'#E67E22','Count':'#27AE60'}
styles      = {'TF-IDF':'-o','BM25':'-s','Count':'-^'}

for mname in ['TF-IDF','BM25','Count']:
    axes[0].plot(K_values_ndcg, ndcg_curves[mname], styles[mname],
                 color=colors_ndcg[mname], label=mname, linewidth=2.5, markersize=8)

axes[0].set_title('NDCG@K — TF-IDF vs BM25 vs CountVectorizer',
                  fontsize=12, fontweight='bold')
axes[0].set_xlabel('K'); axes[0].set_ylabel('NDCG')
axes[0].legend(); axes[0].grid(alpha=0.3)
axes[0].spines['top'].set_visible(False); axes[0].spines['right'].set_visible(False)

# Résumé barres NDCG@10
vals_ndcg = [ndcg_curves[m][K_values_ndcg.index(10)] for m in ['TF-IDF','BM25','Count']]
bars_n = axes[1].bar(['TF-IDF','BM25','Count'], vals_ndcg,
                     color=[colors_ndcg[m] for m in ['TF-IDF','BM25','Count']], alpha=0.85)
axes[1].set_title('NDCG@10 par modèle', fontsize=12, fontweight='bold')
axes[1].set_ylim(0, max(vals_ndcg)*1.25)
for bar, v in zip(bars_n, vals_ndcg):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                 f'{v:.4f}', ha='center', fontsize=11, fontweight='bold')
axes[1].spines['top'].set_visible(False); axes[1].spines['right'].set_visible(False)

plt.suptitle('NDCG@K — Normalized Discounted Cumulative Gain',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig7_ndcg.png', dpi=150, bbox_inches='tight')
plt.show()
print('📸 fig7_ndcg.png sauvegardé')

## 📊 17. Tableau comparatif final — Toutes métriques

In [ ]:
# Tableau final complet : Precision + Recall + NDCG + Coverage + Personalization
df_all = df_ndcg.copy()
df_all['Coverage (%)']        = df_k10['Coverage (%)']
df_all['Personalization (%)'] = df_k10['Personalization (%)']

# Score composite avec NDCG
df_all['Score_Composite'] = (
    0.30 * df_all['Precision@10'] +
    0.25 * df_all['Recall@10'] +
    0.25 * df_all['NDCG@10'] +
    0.10 * df_all['Coverage (%)']        / 100 +
    0.10 * df_all['Personalization (%)'] / 100
).round(4)

print('=== TABLEAU COMPARATIF COMPLET ===')
display(df_all.style
        .highlight_max(color='#d4edda')
        .highlight_min(color='#f8d7da')
        .format(precision=4))

best_final = df_all['Score_Composite'].idxmax()
print(f'\n🏆 Meilleur modèle : {best_final}')
print(f'   Precision@10    : {df_all.loc[best_final,"Precision@10"]:.4f}')
print(f'   Recall@10       : {df_all.loc[best_final,"Recall@10"]:.4f}')
print(f'   NDCG@10         : {df_all.loc[best_final,"NDCG@10"]:.4f}')
print(f'   Coverage        : {df_all.loc[best_final,"Coverage (%)"]}%')
print(f'   Personalization : {df_all.loc[best_final,"Personalization (%)"]}%')